<a href="https://colab.research.google.com/github/AW-OMW/FLY-RANK-PROJECT/blob/main/notebooks/02_your_first_readable_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AW-OMW/FLY-RANK-PROJECT/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [11]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [12]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [13]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [22]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- class: 0
|   |--- avg_position >  0.15
|   |   |--- class: 0
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- class: 1
|   |--- content_age_days >  287.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [15]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.550
Precision@50:  hand rule 0.680   vs   tree 0.600


Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [16]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [20]:
# Your experiment here changing depth of split tree to 3 first and then 4
from sklearn.tree import DecisionTreeClassifier, export_text

# Define and fit tree with max_depth=3
tree_depth3 = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree_depth3.fit(X, y)

print("--- Decision Tree (max_depth=3) ---")
print(export_text(tree_depth3, feature_names=features))

tree_score_depth3 = tree_depth3.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr_depth3 = precision_at_k(tree_score_depth3, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree (depth 3) {tr_depth3:.3f}")

print("\n--- Decision Tree (max_depth=4) ---")
# Define and fit tree with max_depth=4
tree_depth4 = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
tree_depth4.fit(X, y)

print(export_text(tree_depth4, feature_names=features))

tree_score_depth4 = tree_depth4.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr_depth4 = precision_at_k(tree_score_depth4, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree (depth 4) {tr_depth4:.3f}")

--- Decision Tree (max_depth=3) ---
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- content_age_days <= 108.50
|   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- avg_position <= 25.15
|   |   |   |--- class: 0
|   |   |--- avg_position >  25.15
|   |   |   |--- class: 0

Precision@20:  hand rule 0.900   vs   tree (depth 3) 0.700
Precision@50:  hand rule 0.680   vs   tree (depth 3) 0.720

--- Decision Tree (max_depth=4) ---
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- word_count <= 687.00
| 

In [21]:
import pandas as pd

feature_importances_depth3 = pd.Series(tree_depth3.feature_importances_, index=features).sort_values(ascending=False)
feature_importances_depth4 = pd.Series(tree_depth4.feature_importances_, index=features).sort_values(ascending=False)

print("Feature Importances (max_depth=3):")
display(feature_importances_depth3)

print("\nFeature Importances (max_depth=4):")
display(feature_importances_depth4)

# For a side-by-side comparison
comparison_df = pd.DataFrame({
    'Depth_3': feature_importances_depth3,
    'Depth_4': feature_importances_depth4
}).fillna(0)

print("\nSide-by-side Comparison of Feature Importances:")
display(comparison_df.sort_values(by='Depth_4', ascending=False))

Feature Importances (max_depth=3):


,0
impressions_90d,0.563911
content_age_days,0.251026
avg_position,0.114519
ctr,0.070544
days_since_last_update,0.000000
word_count,0.000000



Feature Importances (max_depth=4):


,0
impressions_90d,0.585140
content_age_days,0.221195
avg_position,0.100619
ctr,0.061982
word_count,0.028561
days_since_last_update,0.002503



Side-by-side Comparison of Feature Importances:


,Depth_3,Depth_4
impressions_90d,0.563911,0.585140
content_age_days,0.251026,0.221195
avg_position,0.114519,0.100619
ctr,0.070544,0.061982
word_count,0.000000,0.028561
days_since_last_update,0.000000,0.002503


In [23]:
print("Current features list:")
print(features)
print("\nShape of X with new features:")
print(X.shape)
print("\nHead of X to show new features:")
display(X.head())

Current features list:
['content_age_days', 'days_since_last_update', 'engagement_rate', 'avg_position', 'ctr', 'word_count']

Shape of X with new features:
(30000, 6)

Head of X to show new features:


,content_age_days,days_since_last_update,engagement_rate,avg_position,ctr,word_count
0,187,20,5.88,10.6,0.76,3221.0
1,445,25,0.00,20.3,0.05,2481.0
2,141,20,0.00,36.5,0.09,3515.0
3,463,22,1.28,6.2,0.49,0.0
4,263,14,0.00,44.0,0.13,2803.0


In [24]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
import pandas as pd

# Perform a train-test split (e.g., 80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("--- Train/Test Split Models ---")

# Define and fit tree with max_depth=3 on training data
tree_depth3_tt = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree_depth3_tt.fit(X_train, y_train)

print("\n--- Decision Tree (max_depth=3, trained on split) ---")
print(export_text(tree_depth3_tt, feature_names=features))

# Evaluate on test data
tree_score_depth3_tt = tree_depth3_tt.predict_proba(X_test)[:, 1]
print("\nPrecision on Test Data (Depth 3):")
for k in (20, 50):
    # Need hand_rule_score for the test set too
    # Assuming df_test can be created from original df for comparison if needed
    # For simplicity, let's just compare tree_depth3_tt against itself for now for scores,
    # but for a true comparison with hand rule, hand rule scores for X_test would be needed
    # Re-calculate hand_rule_score for the test set if we want to compare with it accurately
    # For this task, we'll focus on tree performance on test data
    tr_depth3_tt = precision_at_k(tree_score_depth3_tt, y_test, k)
    print(f"Precision@{k}:   tree (depth 3) {tr_depth3_tt:.3f}")

# Define and fit tree with max_depth=4 on training data
tree_depth4_tt = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
tree_depth4_tt.fit(X_train, y_train)

print("\n--- Decision Tree (max_depth=4, trained on split) ---")
print(export_text(tree_depth4_tt, feature_names=features))

# Evaluate on test data
tree_score_depth4_tt = tree_depth4_tt.predict_proba(X_test)[:, 1]
print("\nPrecision on Test Data (Depth 4):")
for k in (20, 50):
    tr_depth4_tt = precision_at_k(tree_score_depth4_tt, y_test, k)
    print(f"Precision@{k}:   tree (depth 4) {tr_depth4_tt:.3f}")

# Compare Feature Importances
print("\n--- Feature Importance Comparison (Train/Test Split) ---")
feature_importances_depth3_tt = pd.Series(tree_depth3_tt.feature_importances_, index=features).sort_values(ascending=False)
feature_importances_depth4_tt = pd.Series(tree_depth4_tt.feature_importances_, index=features).sort_values(ascending=False)

print("Feature Importances (max_depth=3, Train/Test):")
display(feature_importances_depth3_tt)

print("\nFeature Importances (max_depth=4, Train/Test):")
display(feature_importances_depth4_tt)

comparison_df_tt = pd.DataFrame({
    'Depth_3_TT': feature_importances_depth3_tt,
    'Depth_4_TT': feature_importances_depth4_tt
}).fillna(0)

print("\nSide-by-side Comparison of Feature Importances (Train/Test):")
display(comparison_df_tt.sort_values(by='Depth_4_TT', ascending=False))

--- Train/Test Split Models ---

--- Decision Tree (max_depth=3, trained on split) ---
|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- word_count <= 669.50
|   |   |   |--- class: 0
|   |   |--- word_count >  669.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.15
|   |   |--- ctr <= 0.04
|   |   |   |--- class: 0
|   |   |--- ctr >  0.04
|   |   |   |--- class: 0
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  287.50
|   |   |--- avg_position <= 39.25
|   |   |   |--- class: 0
|   |   |--- avg_position >  39.25
|   |   |   |--- class: 0


Precision on Test Data (Depth 3):
Precision@20:   tree (depth 3) 0.500
Precision@50:   tree (depth 3) 0.660

--- Decision Tree (max_depth=4, trained on split) ---
|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- word_count <= 669.50
|   |   |   |--- engagement_

,0
avg_position,0.594481
content_age_days,0.348841
ctr,0.056016
word_count,0.000662
days_since_last_update,0.000000
engagement_rate,0.000000



Feature Importances (max_depth=4, Train/Test):


,0
avg_position,0.516908
content_age_days,0.332316
ctr,0.070874
days_since_last_update,0.058871
word_count,0.019852
engagement_rate,0.001179



Side-by-side Comparison of Feature Importances (Train/Test):


,Depth_3_TT,Depth_4_TT
avg_position,0.594481,0.516908
content_age_days,0.348841,0.332316
ctr,0.056016,0.070874
days_since_last_update,0.000000,0.058871
word_count,0.000662,0.019852
engagement_rate,0.000000,0.001179


### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.